In [36]:
import os as os
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError
from lxml_html_clean import clean_html
import csv
import logging
import time
from multiprocessing.pool import ThreadPool
import threading
from sys import stdin, stdout
from time import perf_counter
from typing import Iterator, List, Optional

import pandas as pd 
import numpy as np 

import requests
from habanero import Crossref

from metapub import PubMedFetcher, PubMedArticle
from metapub import pubmedcentral


from sqlitedict import SqliteDict

In [37]:

# Define NCBI error handling decorator with tenacity
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms
    retry=retry_if_exception_type([EutilsNCBIError, EutilsRequestError])
)()

@retry_on_communication_error
def get_list(query):
    """
    Retrieve all PMIDs for a given query using the PubMedFetcher.
    """
    fetch = PubMedFetcher()
    num_of_articles = 500
    start_index = 0
    pmids = []
    while True:
        pmid_batch = fetch.pmids_for_query(query,
                                        retstart=start_index,
                                        retmax=num_of_articles,
                                        pmc_only=False)
        pmids.extend(pmid_batch)
        start_index = len(pmids)
        if len(pmid_batch) < num_of_articles:
            break
    return pmids

def read_query_from_file(filename):
    """
    Read and clean query from a file.
    """
    try:
        with open(filename, 'r') as file:
            query = file.read()
        return query
    except FileNotFoundError:
        logging.error(f"The file '{filename}' was not found.")
        return ""
    except Exception as e:
        logging.error(f"An error occurred while reading the query file: {e}")
        return ""

def fetch_pmids_over_period(query_file, start="2000-01-01", stop=None):
    """
    Fetch PMIDs over a specified period using a query read from a file.
    """
    query = read_query_from_file(query_file)
    if not query:
        logging.error("Failed to read query.")
        return np.array([])

    if stop is None:
        stop = datetime.now().strftime("%Y-%m-%d")

    start_date_str = start
    pmid_list = []

    a = datetime.now()

    while True:
        if date.fromisoformat(start_date_str) <= date.fromisoformat("2002-07-01"):
            month_interval = 6
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2005-11-01"):
            month_interval = 5
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2009-11-01"):
            month_interval = 4
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2011-10-01"):
            month_interval = 3
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2023-01-01"):
            month_interval = 2
        else:
            month_interval = 4

        next_start = date.fromisoformat(start_date_str) + relativedelta(months=month_interval)
        end_date = (next_start - relativedelta(days=1))
        end_date_str = end_date.strftime('%Y-%m-%d')

        date_str = f'''(("{start_date_str}"[Date - Publication] : "{end_date_str}"[Date - Publication]) '''
        pmids = get_list(date_str + query)
        pmid_list.extend(pmids)
        start_date_str = next_start.strftime('%Y-%m-%d')
        if next_start >= date.fromisoformat(stop):
            break

    # Remove duplicates by converting to a set, then back to a list
    pmid_clean_list = list(set(pmid_list))
    logging.info(f"Total query duration: {datetime.now() - a}")
    logging.info(f"Total PMIDs fetched: {len(pmid_clean_list)}")

    return np.array(pmid_clean_list)

def save_pmids(pmid_array, directory="PMID_lists"):
    """
    Save PMIDs to both a text file and a NumPy binary file.
    """
    # Ensure the directory exists
    os.makedirs(directory, exist_ok=True)

    # Create a date tag for the filename
    date_tag = datetime.now().isoformat()[:10]

    # File paths
    txt_file_path = os.path.join(directory, f'pmids{date_tag}.txt')
    npy_file_path = os.path.join(directory, f'pmids{date_tag}.npy')

    # Save PMIDs to a text file
    np.savetxt(txt_file_path, pmid_array, fmt='%s', delimiter=",")
    logging.info(f"PMIDs saved to text file: {txt_file_path}")

    # Save PMIDs to a NumPy binary file
    np.save(npy_file_path, pmid_array)
    logging.info(f"PMIDs saved to binary file: {npy_file_path}")


In [ ]:

def main():
    # Define the query file and dates
    query_file = "query"
    start_date = "2000-01-01"

    # Fetch PMIDs over the specified period
    pmid_array = fetch_pmids_over_period(query_file, start=start_date)

    # Save PMIDs if any are fetched
    if len(pmid_array) > 0:  # Use len() to check the number of elements in the list
        save_pmids(pmid_array)
    else:
        print("No PMIDs were fetched.")

if __name__ == "__main__":
    main()
